# Lab 1 - Machine Learning Project
**End-to-End project: From Raw Data to model.pkl**

In [ ]:
import sys
import sklearn
import numpy as np
import pandas as pd
import os
import matplotlib.pyplot as plt
import joblib

# Scikit-Learn >=0.20 is required
assert sklearn.__version__ >= "0.20"

# Matplotlib setup
%matplotlib inline

## 1. Data Collection
We will use the Housing dataset already present in the workspace.

In [ ]:
HOUSING_PATH = os.path.join("handson-ml2-vn-main", "datasets", "housing")

def load_housing_data(housing_path=HOUSING_PATH):
    csv_path = os.path.join(housing_path, "housing.csv")
    return pd.read_csv(csv_path)

housing = load_housing_data()
housing.head()

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# General info
housing.info()

# Histogram
housing.hist(bins=50, figsize=(20,15))
plt.show()

# Scatter plot of coordinates
housing.plot(kind="scatter", x="longitude", y="latitude", alpha=0.4,
             s=housing["population"]/100, label="population", figsize=(10,7),
             c="median_house_value", cmap=plt.get_cmap("jet"), colorbar=True,
             sharex=False)
plt.legend()
plt.show()

# Correlation matrix
corr_matrix = housing.corr(numeric_only=True)
print("\nCorrelation with median_house_value:")
print(corr_matrix["median_house_value"].sort_values(ascending=False))

## 3. Data Preparation
Handling missing values, encoding, and scaling.

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

# Create a stratified split based on income category
housing["income_cat"] = pd.cut(housing["median_income"],
                               bins=[0., 1.5, 3.0, 4.5, 6.0, np.inf],
                               labels=[1, 2, 3, 4, 5])

split = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
for train_index, test_index in split.split(housing, housing["income_cat"]):
    strat_train_set = housing.loc[train_index]
    strat_test_set = housing.loc[test_index]

for set_ in (strat_train_set, strat_test_set):
    set_.drop("income_cat", axis=1, inplace=True)

housing = strat_train_set.drop("median_house_value", axis=1)
housing_labels = strat_train_set["median_house_value"].copy()

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# Separate numerical and categorical columns
housing_num = housing.drop("ocean_proximity", axis=1)
num_attribs = list(housing_num)
cat_attribs = ["ocean_proximity"]

# Numerical pipeline: Handle missing values and scale
num_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("std_scaler", StandardScaler()),
    ])

# Full pipeline: Combine numerical and categorical pipelines
full_pipeline = ColumnTransformer([
        ("num", num_pipeline, num_attribs),
        ("cat", OneHotEncoder(), cat_attribs),
    ])

housing_prepared = full_pipeline.fit_transform(housing)
print("Prepared data shape:", housing_prepared.shape)

## 4. Training Models
Training Linear Regression, Decision Tree, and Random Forest.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# Linear Regression
lin_reg = LinearRegression()
lin_reg.fit(housing_prepared, housing_labels)
housing_predictions = lin_reg.predict(housing_prepared)
lin_rmse = np.sqrt(mean_squared_error(housing_labels, housing_predictions))
print("Linear Regression RMSE:", lin_rmse)

# Decision Tree
tree_reg = DecisionTreeRegressor(random_state=42)
tree_reg.fit(housing_prepared, housing_labels)
housing_predictions = tree_reg.predict(housing_prepared)
tree_rmse = np.sqrt(mean_squared_error(housing_labels, housing_predictions))
print("Decision Tree RMSE:", tree_rmse)

# Random Forest
forest_reg = RandomForestRegressor(n_estimators=100, random_state=42)
forest_reg.fit(housing_prepared, housing_labels)
housing_predictions = forest_reg.predict(housing_prepared)
forest_rmse = np.sqrt(mean_squared_error(housing_labels, housing_predictions))
print("Random Forest RMSE:", forest_rmse)

## 5. Model Tuning (Grid Search)

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = [
    {"n_estimators": [3, 10, 30], "max_features": [2, 4, 6, 8]},
    {"bootstrap": [False], "n_estimators": [3, 10], "max_features": [2, 3, 4]},
]

forest_reg = RandomForestRegressor(random_state=42)
grid_search = GridSearchCV(forest_reg, param_grid, cv=5,
                           scoring="neg_mean_squared_error",
                           return_train_score=True)

grid_search.fit(housing_prepared, housing_labels)

print("Best parameters:", grid_search.best_params_)

# Evaluate on test set
final_model = grid_search.best_estimator_

X_test = strat_test_set.drop("median_house_value", axis=1)
y_test = strat_test_set["median_house_value"].copy()

X_test_prepared = full_pipeline.transform(X_test)
final_predictions = final_model.predict(X_test_prepared)

final_rmse = np.sqrt(mean_squared_error(y_test, final_predictions))
print("Final Model RMSE on Test Set:", final_rmse)

## 6. Saving the Model

In [ ]:
# Create a full pipeline that includes both preparation and the final prediction model
full_pipeline_with_predictor = Pipeline([
        ("preparation", full_pipeline),
        ("final_model", final_model)
    ])

# Train the full pipeline on the full dataset (optional but recommended for production)
full_pipeline_with_predictor.fit(housing, housing_labels)

joblib.dump(full_pipeline_with_predictor, "model.pkl")
print("Model successfully saved to model.pkl")

# Verification: Load the model
loaded_model = joblib.load("model.pkl")
print("Model successfully loaded.")